## Notebook to learn to play with tif images

In [ ]:
import os
import sys
import random
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchinfo
import pprint

from utils import utils
from data_builder import build_tags
from data_builder import data_loader
from model_builder import build_model
import trainer.metrics as metrics_module

from trainer.trainer import Trainer

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"pytorch version = {torch.__version__}")

In [ ]:
# GET config
EXP_NAME = "exp_001"
config = utils.get_config(EXP_NAME)
config["mode"] = "training"
# display(config)

# SET RANDOM SEEDS
torch.manual_seed(config["seed"])
torch.cuda.manual_seed(config["seed"])
np.random.seed(config["seed"])
random.seed(config["seed"])
torch.backends.cudnn.deterministic = True

# SET DIRECTORIES
directory_paths = utils.get_directories()
FIGURE_DIRECTORY = directory_paths["figures_dir"]
MODEL_DIRECTORY = directory_paths["save_model_dir"]

In [ ]:
# LOAD THE DATA
tags_train, tags_val = build_tags.get_tags(config)

ds_train = data_loader.CustomData(config, tags_train)
train_loader = torch.utils.data.DataLoader(
    ds_train,
    batch_size=None,
    batch_sampler=None,
    shuffle=True,
    drop_last=False,
)

ds_val = data_loader.CustomData(config, tags_val)
val_loader = torch.utils.data.DataLoader(
    ds_val,
    batch_size=None,
    batch_sampler=None,
    shuffle=False,
    drop_last=False,
)

x, y = train_loader.dataset[0]
print(f"{x.shape = }")
print(f"{y.shape = }")

In [ ]:
# BUILD THE MODEL
model = build_model.TorchModel(config)

# print the model summary
input_shape = (
    64,
    len(config["channels"]),
    config["scene_width_landsat"],
    config["scene_width_landsat"],
)
# torchinfo.summary(
#     model,
#     input_shape,
#     verbose=1,
#     col_names=("input_size", "output_size", "num_params"),
# )

In [ ]:
# Setup the optimizer, loss function, and metrics
optimizer = getattr(torch.optim, config["optimizer"]["type"])(
    model.parameters(), **config["optimizer"]["args"]
)
criterion = torch.nn.MSELoss()
metric_funcs = [getattr(metrics_module, met) for met in config["metrics"]]

# BUILD THE TRAINER
device = utils.prepare_device(config["device"])
trainer = Trainer(
    model,
    criterion,
    metric_funcs,
    optimizer,
    max_epochs=config["trainer"]["max_epochs"],
    data_loader=train_loader,
    validation_data_loader=val_loader,
    device=device,
    config=config,
)

In [ ]:
# FIT THE MODEL

# Train the Model
model.to(device)
trainer.fit()
model.eval()

# Save the Pytorch Model
model_name = utils.get_model_name(config["exp_name"], config["seed"])
utils.save_torch_model(model, MODEL_DIRECTORY + model_name + ".pt")


In [ ]:
print(trainer.log.history.keys())

plt.figure(figsize=(20, 4))
for i, m in enumerate(("loss", *config["metrics"])):
    plt.subplot(1, 4, i + 1)
    plt.plot(trainer.log.history["epoch"], trainer.log.history[m], label=m)
    plt.plot(
        trainer.log.history["epoch"], trainer.log.history["val_" + m], label="val_" + m
    )
    plt.axvline(
        x=trainer.early_stopper.best_epoch, linestyle="--", color="k", linewidth=0.75
    )
    plt.title(m)
    plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("training complete.")